# KnottedGraph vs Topoly: identical-PD Yamada performance

This is the user-facing performance comparison. **Both frameworks receive the same planar-diagram (PD) input.** The benchmark does not use Topoly's coordinate-plus-bridges convenience path.

Correctness is checked before any timing is accepted. The comparison permits only the standard Laurent convention relation

$$P_{\mathrm{Topoly}}(A)=\pm A^k P_{\mathrm{KG}}(A^{\pm1}),$$

which accounts for a global Laurent unit and the conventional $A\leftrightarrow A^{-1}$ variable orientation. Any discrepancy beyond that aborts the benchmark. Topoly's global Yamada memo table is cleared before each timed evaluation.

The plots below show scaling with **crossing number**, **edge number**, and **vertex number**. The connected suite is especially important because it prevents KnottedGraph's independent-diagram block factorization from being the sole source of a speed advantage.

In [ ]:
from pathlib import Path
import csv
import json
import os
import subprocess
import sys

import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
if not (ROOT / 'src' / 'knotted_graph').exists():
    raise RuntimeError('Run from inside the KnottedGraph checkout.')
SRC = ROOT / 'src'
sys.path.insert(0, str(SRC))
branch = subprocess.check_output(['git','rev-parse','--abbrev-ref','HEAD'], cwd=ROOT, text=True).strip()
commit = subprocess.check_output(['git','rev-parse','HEAD'], cwd=ROOT, text=True).strip()
print('ROOT   =', ROOT)
print('branch =', branch)
print('commit =', commit)
if branch != 'perf/yamada-max-optimization':
    raise RuntimeError(f'Expected perf/yamada-max-optimization, got {branch}')

import knotted_graph
kg_path = Path(knotted_graph.__file__).resolve()
if SRC not in kg_path.parents:
    raise RuntimeError(f'A stale knotted_graph was imported from {kg_path}')
try:
    import topoly
except ImportError as exc:
    raise ImportError('Install Topoly first: pip install topoly') from exc
print('knotted_graph =', kg_path)
print('topoly =', Path(topoly.__file__).resolve())

OUT = ROOT / 'User_guide' / 'benchmarks'
RES = OUT / 'results_latest'
FIG = OUT / 'figures_latest'
RES.mkdir(exist_ok=True)
FIG.mkdir(exist_ok=True)


## 1. Run correctness-gated identical-PD benchmarks

In [ ]:
def run_summary(script, timeout=3600):
    path = ROOT / 'dev' / script
    if not path.exists():
        raise FileNotFoundError(path)
    env = dict(os.environ)
    env['PYTHONPATH'] = str(SRC)
    env['PYTHONNOUSERSITE'] = '1'
    proc = subprocess.run(
        [sys.executable, str(path)], cwd=ROOT, env=env,
        text=True, capture_output=True, timeout=timeout,
    )
    if proc.stdout:
        print(proc.stdout)
    if proc.returncode:
        raise RuntimeError(
            f'{script} failed with exit code {proc.returncode}.\n'
            f'STDOUT:\n{proc.stdout}\nSTDERR:\n{proc.stderr}'
        )
    for line in proc.stdout.splitlines():
        if line.startswith('SUMMARY='):
            return json.loads(line[8:])
    raise RuntimeError(f'{script} completed but did not emit SUMMARY=')

decomposable = run_summary('benchmark_topoly_identical_pd.py')
connected = run_summary('benchmark_topoly_connected_pd.py')
print('decomposable rows =', len(decomposable))
print('connected rows    =', len(connected))


In [ ]:
def save_csv(name, rows):
    keys = list(dict.fromkeys(key for row in rows for key in row))
    with (RES / name).open('w', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=keys)
        writer.writeheader()
        writer.writerows(rows)

save_csv('topoly_identical_pd.csv', decomposable)
save_csv('topoly_connected_pd.csv', connected)


## 2. Scaling with projected crossing number

The decomposable series isolates the crossing-state scaling in a controlled family. Connected cases are overlaid as independent points so the comparison is not restricted to factorable diagrams.

In [ ]:
plt.figure(figsize=(8.5, 5.4))
q = sorted(decomposable, key=lambda row: row['crossings'])
plt.plot([r['crossings'] for r in q], [r['knottedgraph_s'] for r in q], marker='o', label='KnottedGraph — decomposable')
plt.plot([r['crossings'] for r in q], [r['topoly_s'] for r in q], marker='o', label='Topoly — decomposable')
plt.scatter([r['crossings'] for r in connected], [r['knottedgraph_s'] for r in connected], marker='x', label='KnottedGraph — connected')
plt.scatter([r['crossings'] for r in connected], [r['topoly_s'] for r in connected], marker='+', label='Topoly — connected')
plt.yscale('log')
plt.xlabel('Projected crossings, c')
plt.ylabel('Yamada evaluation time (s)')
plt.title('Identical-PD Yamada scaling with crossing number')
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.savefig(FIG / 'topoly_vs_knottedgraph_crossings.pdf', bbox_inches='tight')
plt.savefig(FIG / 'topoly_vs_knottedgraph_crossings.png', dpi=300, bbox_inches='tight')
plt.show()


## 3. Scaling with graph edge number

Only the connected suite is used here. Multiple points at the same edge count correspond to different valid projections/crossing counts of the same abstract graph.

In [ ]:
plt.figure(figsize=(8.5, 5.4))
q = sorted(connected, key=lambda row: (row['E'], row['crossings'], row['graph']))
plt.scatter([r['E'] for r in q], [r['knottedgraph_s'] for r in q], marker='o', label='KnottedGraph')
plt.scatter([r['E'] for r in q], [r['topoly_s'] for r in q], marker='x', label='Topoly')
plt.yscale('log')
plt.xlabel('Graph edges, E')
plt.ylabel('Yamada evaluation time (s)')
plt.title('Connected identical-PD scaling with edge number')
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.savefig(FIG / 'topoly_vs_knottedgraph_edges.pdf', bbox_inches='tight')
plt.savefig(FIG / 'topoly_vs_knottedgraph_edges.png', dpi=300, bbox_inches='tight')
plt.show()


## 4. Scaling with graph vertex number

In [ ]:
plt.figure(figsize=(8.5, 5.4))
q = sorted(connected, key=lambda row: (row['V'], row['crossings'], row['graph']))
plt.scatter([r['V'] for r in q], [r['knottedgraph_s'] for r in q], marker='o', label='KnottedGraph')
plt.scatter([r['V'] for r in q], [r['topoly_s'] for r in q], marker='x', label='Topoly')
plt.yscale('log')
plt.xlabel('Graph vertices, V')
plt.ylabel('Yamada evaluation time (s)')
plt.title('Connected identical-PD scaling with vertex number')
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.savefig(FIG / 'topoly_vs_knottedgraph_vertices.pdf', bbox_inches='tight')
plt.savefig(FIG / 'topoly_vs_knottedgraph_vertices.png', dpi=300, bbox_inches='tight')
plt.show()


## 5. Connected-case timing ratios

`Topoly / KnottedGraph > 1` means KnottedGraph is faster for that identical PD input.

In [ ]:
for row in sorted(connected, key=lambda r: (r['crossings'], r['E'], r['V'], r['graph'])):
    print(
        f"{row['graph']:10s} V={row['V']:2d} E={row['E']:2d} c={row['crossings']:2d}  "
        f"KG={row['knottedgraph_s']:.6g}s  Topoly={row['topoly_s']:.6g}s  "
        f"Topoly/KG={row['topoly_over_kg']:.3g}x"
    )

print('\nPASS: every displayed timing passed the polynomial-equivalence gate first.')
